In [ ]:
import pandas as pd
import os
import matplotlib.pyplot as plt

import hashlib
from surprise import Dataset, Reader
from surprise.model_selection import train_test_split
from surprise import SVD
from surprise import accuracy
from surprise.model_selection import GridSearchCV

In [ ]:
df=pd.read_csv("Data/matriz.csv")
df.head()

In [ ]:
df_recomend=pd.read_csv("Data/objetivos_dni.csv")
df_recomend.head(11)

In [ ]:
df_dnis = df_recomend[df_recomend["explica"]=="Otros como tu han comprado"].copy()

In [ ]:
df_codigo_producto = df_recomend[df_recomend["explica"]=="Articulo promocionado"]

In [ ]:
df_dnis.head(11)

In [ ]:
def anonimizar_dni(dni: str) -> str:
    return hashlib.sha256(dni.strip().upper().encode()).hexdigest()

df_dnis["id_cliente_enc"]=df_dnis["id"].apply(anonimizar_dni)

df_dnis.head(10)

In [ ]:
df_dnis["id_cliente_enc"].tolist()

In [ ]:
clientes_recomendar = df_dnis["id_cliente_enc"].tolist()

In [ ]:
clientes_recomendar

In [ ]:
df.shape

In [ ]:
df.info() #SI EL ID DE USUARIO Y EL DE CLIENTE ESTAN EN FORMATOS DIFERENTES NO VA A FUNCIONAR 

In [ ]:
df["cod_producto"]= df["cod_producto"].astype(str)

In [ ]:
df.describe()

In [ ]:
df.isna().sum()

In [ ]:
df.duplicated().sum()

In [ ]:
df_maestro=pd.read_csv("Data/maestroestr.csv")
df_maestro.head()

In [ ]:
min = df["cantidad"].min()
max = df["cantidad"].max()  
reader = Reader(rating_scale=(min, max))

In [ ]:
dataset_antes = df.copy()

In [ ]:
dataset_antes

In [ ]:

df = Dataset.load_from_df(df[["id_cliente_enc", "cod_producto", "cantidad"]], reader)

In [ ]:
train_set, test_set = train_test_split(df, test_size=0.25, random_state=42)

In [ ]:
svd= SVD(n_factors=100)
svd.fit(train_set)

In [ ]:
predicciones = svd.test(test_set)

df_train = df.build_full_trainset() #ahora entrenamos todo el df

In [ ]:
mae = accuracy.mae(predicciones)

In [ ]:
rmse= accuracy.rmse(predicciones)


In [ ]:
puntuacion_actual = [(pred.uid, pred.iid, pred.r_ui) for pred in predicciones]
puntuacion_predicha = [(pred.uid, pred.iid, pred.est) for pred in predicciones]

In [ ]:
df_actual = pd.DataFrame(puntuacion_actual, columns=["id_cliente_enc", "cod_producto", "cantidad_actual"])

In [ ]:
df_predicho = pd.DataFrame(puntuacion_predicha, columns=["id_cliente_enc", "cod_producto", "cantidad_predicha"])

In [ ]:
df_combinado = pd.merge(df_actual, df_predicho, on = ["id_cliente_enc", "cod_producto"])

In [ ]:
df_combinado

In [ ]:
todos_productos = df_combinado["cod_producto"].unique()


resultados = []

for cliente in clientes_recomendar:
    ya_comprados = dataset_antes[dataset_antes["id_cliente_enc"] == cliente]["cod_producto"].astype(str).tolist()
    predicciones = [svd.predict(cliente, prod) for prod in todos_productos if prod not in ya_comprados]
    top1 = sorted(predicciones, key=lambda x: x.est, reverse=True)[:1]
    for pred in top1:
        resultados.append({
            "id_cliente": cliente,
            "cod_producto": pred.iid,
            "score": pred.est
        })

df_recomendaciones = pd.DataFrame(resultados)
df_recomendaciones
print(ya_comprados)

In [ ]:
df_recomendaciones["cod_producto"] = df_recomendaciones["cod_producto"].astype(str)
df_maestro["cod_est"] = df_maestro["cod_est"].astype(str)

df_final = df_recomendaciones.merge(df_maestro, left_on="cod_producto", right_on="cod_est").drop(columns="cod_est")
df_final.sort_values("id_cliente").reset_index(drop=True)

In [ ]:
lista_recomendar = df_codigo_producto["id"].tolist()
print(lista_recomendar)

In [ ]:
def recomendar_clientes_para_producto(cod_producto, n=10):
    todos_clientes = dataset_antes["id_cliente_enc"].unique()
    ya_compraron = dataset_antes[dataset_antes["cod_producto"] == cod_producto]["id_cliente_enc"].tolist()
    predicciones = [svd.predict(cliente, cod_producto, clip=True) 
                    for cliente in todos_clientes if cliente not in ya_compraron]
    top_n = sorted(predicciones, key=lambda x: x.est, reverse=True)[:n]
    return pd.DataFrame([{"id_cliente": pred.uid, "score": round(pred.est, 4)} for pred in top_n])

# Usar la función
df_top10 = recomendar_clientes_para_producto('04201510')
df_top10